In [ ]:
def is_variable(term):
    return isinstance(term, str) and len(term) == 1 and term.islower()


def unify(x, y, subs):
    if x == y:
        return subs

    if is_variable(x):
        return unify_var(x, y, subs)

    if is_variable(y):
        return unify_var(y, x, subs)

    if isinstance(x, tuple) and isinstance(y, tuple):
        if x[0] != y[0] or len(x[1]) != len(y[1]):
            return None

        for a, b in zip(x[1], y[1]):
            subs = unify(a, b, subs)
            if subs is None:
                return None

        return subs

    return None


def unify_var(var, term, subs):
    if var in subs:
        return unify(subs[var], term, subs)

    if is_variable(term) and term in subs:
        return unify(var, subs[term], subs)

    if occurs(var, term, subs):
        return None

    subs[var] = term
    return subs


def occurs(var, term, subs):
    if var == term:
        return True

    if isinstance(term, tuple):
        return any(occurs(var, arg, subs) for arg in term[1])

    if is_variable(term) and term in subs:
        return occurs(var, subs[term], subs)

    return False


def parse(expr):
    name, rest = expr.replace(" ", "").split("(", 1)
    args = rest[:-1].split(",")

    return (name, tuple(
        parse(arg) if "(" in arg else arg
        for arg in args
    ))


def substitute(term, subs):
    if is_variable(term) and term in subs:
        return substitute(subs[term], subs)

    if isinstance(term, tuple):
        return (term[0], tuple(
            substitute(arg, subs) for arg in term[1]
        ))

    return term


def display(term):
    if isinstance(term, tuple):
        return term[0] + "(" + ", ".join(
            display(arg) for arg in term[1]
        ) + ")"
    return term


def main():
    cases = [
        ("Eats(x, Apple)", "Eats(Riya, y)"),
        ("Knows(John, x)", "Likes(John, y)")
    ]

    for i, (e1, e2) in enumerate(cases, 1):
        print("\nCase", i)
        print("Expression 1:", e1)
        print("Expression 2:", e2)

        result = unify(parse(e1), parse(e2), {})

        if result is None:
            print("Unification failed.")
        else:
            print("Substitutions:")
            for var, value in result.items():
                print(var, "=", display(value))

            print("Unified Expression:",
                  display(substitute(parse(e1), result)))
            print("Unification successful.")


if __name__ == "__main__":
    main()


Case 1
Expression 1: Eats(x, Apple)
Expression 2: Eats(Riya, y)
Substitutions:
x = Riya
y = Apple
Unified Expression: Eats(Riya, Apple)
Unification successful.

Case 2
Expression 1: Knows(John, x)
Expression 2: Likes(John, y)
Unification failed.


In [ ]:
def is_variable(term):
    return isinstance(term, str) and len(term) == 1 and term.islower()


def match(pattern, fact, theta):
    if len(pattern) != len(fact) or pattern[0] != fact[0]:
        return None

    theta = theta.copy()

    for p, f in zip(pattern[1:], fact[1:]):
        if is_variable(p):
            if p in theta and theta[p] != f:
                return None
            theta[p] = f
        elif p != f:
            return None

    return theta


def find_substitutions(conditions, facts, theta=None):
    if theta is None:
        theta = {}

    if not conditions:
        return [theta]

    results = []

    for fact in facts:
        new_theta = match(conditions[0], fact, theta)

        if new_theta is not None:
            results.extend(
                find_substitutions(
                    conditions[1:], facts, new_theta
                )
            )

    return results


def forward_chaining(facts, rules, query):
    facts = list(facts)

    changed = True
    while changed:
        changed = False

        for conditions, conclusion in rules:
            substitutions = find_substitutions(
                conditions, facts
            )

            for theta in substitutions:
                new_fact = tuple(
                    theta.get(term, term)
                    if is_variable(term) else term
                    for term in conclusion
                )

                if new_fact not in facts:
                    facts.append(new_fact)
                    changed = True
                    print("Inferred:", new_fact)

                if new_fact == query:
                    return True

        if query in facts:
            return True

    return query in facts


def main():
    rules = [
        ([("Missile", "x")],
         ("Weapon", "x")),

        ([("Enemy", "x", "America")],
         ("Hostile", "x")),

        ([("Missile", "x"), ("Owns", "A", "x")],
         ("Sells", "Robert", "x", "A")),

        ([("American", "p"), ("Weapon", "q"),
          ("Sells", "p", "q", "r"), ("Hostile", "r")],
         ("Criminal", "p"))
    ]

    query = ("Criminal", "Robert")

    # Case 1: All required facts are available
    facts1 = [
        ("American", "Robert"),
        ("Owns", "A", "T1"),
        ("Missile", "T1"),
        ("Enemy", "A", "America")
    ]

    # Case 2: Enemy fact is missing
    facts2 = [
        ("American", "Robert"),
        ("Owns", "A", "T1"),
        ("Missile", "T1")
    ]

    for i, facts in enumerate([facts1, facts2], 1):
        print("\nCASE", i)
        print("Query: Criminal(Robert)")

        if forward_chaining(facts, rules, query):
            print("Result: Query proved.")
        else:
            print("Result: Query cannot be proved.")


if __name__ == "__main__":
    main()



CASE 1
Query: Criminal(Robert)
Inferred: ('Weapon', 'T1')
Inferred: ('Hostile', 'A')
Inferred: ('Sells', 'Robert', 'T1', 'A')
Inferred: ('Criminal', 'Robert')
Result: Query proved.

CASE 2
Query: Criminal(Robert)
Inferred: ('Weapon', 'T1')
Inferred: ('Sells', 'Robert', 'T1', 'A')
Result: Query cannot be proved.
